# 07 · Huấn luyện PRDetect V4 trên HC3

Bật GPU và Internet. Notebook lấy mã từ nhánh `version4` trên GitHub. Mặc định V4 dùng cấu hình Abla1 với `V3_STRUCT=0` và `V3_AUG='none'`. Để khôi phục cấu hình V4 trước đây, đặt `V3_STRUCT=1` và `V3_AUG='swap'`. Tiêu chí chọn checkpoint vẫn dùng validation sạch và validation có swap. ZIP chứa checkpoint EMA, A1 và manifest. Ô cuối chấm HC3 sạch, GPT-3.5 sạch và HC3 Replace 25%.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys
from IPython.display import FileLink, display

WORK = Path('/kaggle/working/prdetect_v4')
REPO = WORK / 'PRDetect-Research'
REPO_URL = 'https://github.com/tphan1420/PRDetect-Research.git'
BRANCH = 'version4'
WORK.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO)],
                   check=True)
else:
    subprocess.run(['git', 'fetch', '--depth', '1', REPO_URL, BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=REPO, check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
print(f'PRDetect {BRANCH}: {commit}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements-v3-eval.txt'),
                'torch-geometric'], check=True)
import spacy
if not spacy.util.is_package('en_core_web_sm'):
    subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)
import torch
assert torch.cuda.is_available(), 'Bật GPU trong Kaggle Settings'
print(torch.cuda.get_device_name(0))

In [ ]:
SEED = 2026
V3_EPOCHS = 10
V3_STRUCT = 0
V3_AUG = 'none'
A1_EPOCHS = 10
BATCH_SIZE = 1
MICRO_BATCH_SIZE = 1
VAL_BATCH_SIZE = 64
A1_BATCH = 4
A1_LR = 0.0001
CHECKPOINT_DIR = WORK / 'checkpoints'
FEATURE_CACHE = WORK / 'feature_cache'

In [ ]:
subprocess.run([sys.executable, 'train_v4.py', '--seed', str(SEED),
                '--output-dir', str(CHECKPOINT_DIR), '--feature-cache', str(FEATURE_CACHE),
                '--v3-epochs', str(V3_EPOCHS), '--a1-epochs', str(A1_EPOCHS),
                '--v3-struct', str(V3_STRUCT), '--v3-aug', V3_AUG,
                '--batch-size', str(BATCH_SIZE), '--micro-batch-size', str(MICRO_BATCH_SIZE),
                '--val-batch-size', str(VAL_BATCH_SIZE), '--a1-batch', str(A1_BATCH), '--a1-lr', str(A1_LR)],
               cwd=REPO, check=True)
run_dir = CHECKPOINT_DIR / f'seed{SEED}'
model_name = json.loads((run_dir / 'v4_train.json').read_text(encoding='utf-8'))['model']
archive = run_dir / f'{model_name}_hc3_seed{SEED}.zip'
print(archive)
display(FileLink(str(archive)))

In [ ]:
OUT = WORK / f'three_tests_seed{SEED}'
subprocess.run([sys.executable, 'v4.py', 'evaluate', '--suite', 'three',
                '--checkpoint-dir', str(CHECKPOINT_DIR), '--seeds', str(SEED), '--batch-size', '1',
                '--device', 'cuda', '--output-dir', str(OUT)], cwd=REPO, check=True)
import pandas as pd
display(pd.read_csv(OUT / 'results_by_seed.csv'))
archive = shutil.make_archive(str(OUT), 'zip', OUT.parent, OUT.name)
display(FileLink(archive))